[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# Train Cricket Ball Detector

---

This notebook trains a YOLOv8 model that finds the ball in broadcast footage filmed from
behind the bowler. Public ball models did not transfer between broadcasts in our tests
(red-ball Test and white-ball ODI footage), so the model is trained on frames from the
footage it will be used on.

Frames come from `examples/cricket/extract_ball_frames.py`, which names every frame after
its delivery. Label the ball in a Roboflow project, then run this notebook. It splits
the data by delivery, so the test score comes from deliveries the model has never seen,
not from neighbouring frames of the training deliveries.

## Configure API keys

- Open your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

## Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

## Imports

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
from IPython.display import Image

## Pull dataset

Generate a version of the labelled project with **no resize** in preprocessing and no
augmentations: the ball is only a few pixels wide, and resizing to 640 px would shrink
it further. Set your workspace, project and version below.

In [ ]:
WORKSPACE = "tanmay-surve"
PROJECT = "cricket-ball-broadcast"
VERSION = 2

!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")

## Split by delivery

Neighbouring frames of one delivery are nearly identical. A random split would put them
in both training and test and overstate the score, so whole deliveries are held out.

In [ ]:
import glob
import re
import shutil

import yaml

def label_path(image_path):
    folder = os.path.dirname(os.path.dirname(image_path))
    name = os.path.splitext(os.path.basename(image_path))[0] + ".txt"
    return os.path.join(folder, "labels", name)


images = sorted(glob.glob(f"{dataset.location}/*/images/*.jpg"))
delivery = lambda path: re.split(r"_f\d{4}", os.path.basename(path))[0]
deliveries = sorted({delivery(path) for path in images})
test = set(deliveries[::5])
val = set(deliveries[2::5])

DATA = f"{HOME}/datasets/ball-by-delivery"
shutil.rmtree(DATA, ignore_errors=True)
counts = {}
for path in images:
    split = "test" if delivery(path) in test else "val" if delivery(path) in val else "train"
    for kind, source in (("images", path), ("labels", label_path(path))):
        os.makedirs(f"{DATA}/{split}/{kind}", exist_ok=True)
        if os.path.exists(source):
            shutil.copy(source, f"{DATA}/{split}/{kind}/")
    counts[split] = counts.get(split, 0) + 1

names = yaml.safe_load(open(f"{dataset.location}/data.yaml"))["names"]
yaml.safe_dump({"path": DATA, "train": "train/images", "val": "val/images",
                "test": "test/images", "names": names}, open(f"{DATA}/data.yaml", "w"))
print(counts)
print("test deliveries:", sorted(test))

## Optional: add public ball datasets

Public ball models did not work on our footage as they were, but their labelled balls
are still useful training examples. This adds a sample of each dataset's images to the
training split only; validation and test stay on held-out deliveries of your own
footage, so the score shows whether the extra data helps there. Both datasets are
openly licensed: credit them if you publish the model.

| dataset | author | license |
|:--------|:-------|:--------|
| [cricket-balls](https://universe.roboflow.com/cricket-rfyd8/cricket-balls-l1us5) | cricket-rfyd8 | CC BY 4.0 |
| [stumps](https://universe.roboflow.com/cardiff-university-moxse/stumps) | cardiff-university-moxse | MIT |

Training time grows with the number of images: about 600 per dataset takes about two
hours on a T4. `EXTRA = []` trains on your own frames only; uncomment the list above it to add them.


In [ ]:
import random

import numpy as np

# EXTRA = [("cricket-rfyd8", "cricket-balls-l1us5", 3), ("cardiff-university-moxse", "stumps", 10)]
EXTRA = []
EXTRA_IMAGES = 600
EPOCHS, PATIENCE = (50, 15) if EXTRA else (100, 30)

ball_class = next(i for i, name in enumerate(names) if name.lower() == "ball")


def ball_lines(path, keep):
    """Ball labels of an image, as boxes of class `ball_class`."""
    lines = []
    for line in open(path).read().splitlines() if os.path.exists(path) else []:
        values = line.split()
        if not values or int(values[0]) not in keep:
            continue
        numbers = np.array([float(v) for v in values[1:]])
        if len(numbers) > 4:  # polygon: use its box
            points = numbers.reshape(-1, 2)
            low, high = points.min(axis=0), points.max(axis=0)
            numbers = np.array([*(low + high) / 2, *(high - low)])
        lines.append(f"{ball_class} " + " ".join(f"{v:.6f}" for v in numbers))
    return lines


random.seed(0)
for workspace, project, version in EXTRA:
    extra = rf.workspace(workspace).project(project).version(version).download(
        "yolov8", location=f"{HOME}/datasets/extra-{project}")
    extra_names = yaml.safe_load(open(f"{extra.location}/data.yaml"))["names"]
    keep = {i for i, name in enumerate(extra_names) if "ball" in name.lower()}
    by_source = {}  # one image per source frame: skip augmented copies
    for path in sorted(glob.glob(f"{extra.location}/*/images/*.jpg")):
        by_source.setdefault(os.path.basename(path).split(".rf.")[0], path)
    picked = random.sample(sorted(by_source.values()), min(EXTRA_IMAGES, len(by_source)))
    for path in picked:
        name = f"{project}_{os.path.splitext(os.path.basename(path))[0]}"
        shutil.copy(path, f"{DATA}/train/images/{name}.jpg")
        lines = ball_lines(label_path(path), keep)
        with open(f"{DATA}/train/labels/{name}.txt", "w") as f:
            f.writelines(line + os.linesep for line in lines)
    print(f"{workspace}/{project}: added {len(picked)} images to train, "
          f"ball classes {[extra_names[i] for i in keep]}")
print("train images:", len(os.listdir(f"{DATA}/train/images")))

## Custom training

`imgsz=1280` keeps the frames at their original width, so the ball stays as large as
possible. For a better model, try `yolov8x.pt` and a smaller batch.

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8m.pt data={DATA}/data.yaml batch=8 epochs={EPOCHS} patience={PATIENCE} imgsz=1280 plots=True

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/results.png', width=600)

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/val_batch0_pred.jpg', width=600)

## Validate custom model on the held-out deliveries

In [ ]:
%cd {HOME}

!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={DATA}/data.yaml split=test imgsz=1280

## How often the ball is found

The pitch map links ball detections into a trajectory and needs the ball in most
frames of its flight. This cell reports, on the held-out deliveries, the share of
labelled balls found within a few pixels, and the number of false detections per frame.

In [ ]:
import numpy as np
from ultralytics import YOLO

def read_boxes(path):
    """Boxes as (x, y, w, h), normalized; polygon labels are turned into their boxes."""
    boxes = []
    if os.path.exists(path):
        for line in open(path).read().splitlines():
            values = [float(v) for v in line.split()[1:]]
            if len(values) == 4:
                boxes.append(values)
            elif len(values) >= 6:
                points = np.array(values).reshape(-1, 2)
                low, high = points.min(axis=0), points.max(axis=0)
                boxes.append([*(low + high) / 2, *(high - low)])
    return boxes


model = YOLO(f"{HOME}/runs/detect/train/weights/best.pt")
labelled = found = false = frames = 0
for path in sorted(glob.glob(f"{DATA}/test/images/*.jpg")):
    result = model(path, imgsz=1280, conf=0.25, verbose=False)[0]
    height, width = result.orig_shape
    predicted = result.boxes.xywh.cpu().numpy()[:, :2]
    labels = label_path(path)
    matched = set()
    for x, y, w, h in read_boxes(labels):
        center = np.array([x * width, y * height])
        tolerance = max(8, 1.5 * max(w * width, h * height))
        distances = np.linalg.norm(predicted - center, axis=1) if len(predicted) else np.array([])
        labelled += 1
        if len(distances) and distances.min() <= tolerance:
            found += 1
            matched.add(int(distances.argmin()))
    false += len(predicted) - len(matched)
    frames += 1

print(f"balls found: {found}/{labelled} ({found / max(labelled, 1):.0%})")
print(f"false detections: {false} over {frames} frames ({false / max(frames, 1):.2f} per frame)")

## Save the model

Download `best.pt` and pass it to `examples/cricket/pitch_map.py` with
`--ball_model data/ball.pt`.

In [ ]:
from google.colab import files

files.download(f"{HOME}/runs/detect/train/weights/best.pt")